# Preprocessing Pipeline: Day and Week Approaches

## Runner Injury Risk Prediction — Lovdal et al. (2021) Replication

This notebook implements and documents the **preprocessing pipeline** for both
temporal approaches (day and week). It transforms raw loaded data into clean,
split datasets ready for modeling.

### Pipeline steps

1. **Sentinel replacement** (day only): −0.01 → 0.0 for rest-day perceived metrics (ADR-007)
2. **Train/test split**: GroupShuffleSplit by athlete ID — no athlete leakage (ADR-006)
3. **Target binarization** (week only): continuous injury score → binary at threshold 0.5 (ADR-002)
4. **Scaling**: StandardScaler fit on training data only — saved for conditional use in modeling
5. **Persistence**: processed splits and scalers saved to `data/processed/`

### Key design decisions

- **Save unscaled data**: tree-based models (RF, XGBoost) do not need scaling.
  Scalers are saved separately for Logistic Regression.
- **Parquet format**: preserves dtypes, smaller files, faster I/O than CSV.
- **Class weighting over SMOTE**: primary imbalance strategy (ADR-003).

Every step connects to an EDA finding from notebooks 01–02 and an ADR decision.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ATHLETE_ID_COL,
    INJURY_COL,
    N_DAY_BLOCKS,
    PROCESSED_DATA_DIR,
    RANDOM_SEED,
    SENTINEL_VALUE,
    WEEK_INJURY_THRESHOLD,
)
from src.data_loading import get_feature_columns, load_day_data, load_week_data
from src.preprocessing.common import (
    get_feature_target_groups,
    get_group_kfold,
    split_train_test,
)
from src.preprocessing.day_preprocessor import (
    fit_scaler,
    handle_sentinel_values,
    transform_scaled,
)
from src.preprocessing.io import load_scaler, load_splits, save_scaler, save_splits
from src.preprocessing.week_preprocessor import binarize_target
from src.utils.logging_config import setup_logging
from src.utils.plotting import INJURY_PALETTE, PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

## 1. Data Loading

We load both raw datasets using the same validated functions from the EDA phase.
The column renaming and schema validation happen automatically inside
`load_day_data()` and `load_week_data()`.

In [ ]:
df_day = load_day_data()
df_week = load_week_data()

day_feature_cols = get_feature_columns(df_day)
week_feature_cols = get_feature_columns(df_week)

print("Day approach:")
print(f"  Shape: {df_day.shape[0]:,} rows × {df_day.shape[1]} columns")
print(f"  Feature columns: {len(day_feature_cols)}")
print("\nWeek approach:")
print(f"  Shape: {df_week.shape[0]:,} rows × {df_week.shape[1]} columns")
print(f"  Feature columns: {len(week_feature_cols)}")

---

## 2. Sentinel Value Treatment (Day Approach)

The day-approach dataset uses **−0.01** as a sentinel value for perceived metrics
(`perceived_exertion`, `perceived_training_success`, `perceived_recovery`) on rest
days — days when the athlete did not train and therefore had no subjective rating
to report.

**Why replace with 0.0?** (ADR-007): If an athlete did not train, their perceived
exertion is zero (no effort), their recovery load is zero (nothing to recover from).
The sentinel −0.01 would introduce artificial signal in scaling and distance-based
operations.

The week approach does **not** have sentinel values — rest days are captured by
the `nr_rest_days` feature and by lower values in other weekly aggregates.

In [ ]:
# Count sentinel values before replacement
perceived_features = ["perceived_exertion", "perceived_training_success", "perceived_recovery"]
sentinel_counts = {}

for feat in perceived_features:
    cols = [f"day_{b}_{feat}" for b in range(N_DAY_BLOCKS)]
    total = (df_day[cols] == SENTINEL_VALUE).sum().sum()
    pct = total / (df_day.shape[0] * len(cols)) * 100
    sentinel_counts[feat] = {"count": total, "pct": f"{pct:.1f}%"}

sentinel_summary = pd.DataFrame(sentinel_counts).T
sentinel_summary.columns = ["Total sentinel cells", "% of cells"]
print("Sentinel value (−0.01) prevalence BEFORE replacement:\n")
print(sentinel_summary.to_string())

In [ ]:
# Save a copy of one column BEFORE replacement for the comparison plot
before_col = "day_0_perceived_exertion"
before_data = df_day[before_col].copy()

# Apply sentinel replacement
df_day = handle_sentinel_values(df_day)

# Verify: zero sentinels remain
remaining = (df_day[day_feature_cols] == SENTINEL_VALUE).sum().sum()
assert remaining == 0, f"Expected 0 sentinels after replacement, found {remaining}"
print(f"Sentinel replacement complete. Remaining sentinel values: {remaining}")

In [ ]:
# Before/after comparison: day_0_perceived_exertion
after_data = df_day[before_col]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax1.hist(before_data, bins=50, color=PALETTE["neutral"], alpha=0.7, edgecolor="white")
ax1.axvline(SENTINEL_VALUE, color=PALETTE["negative"], linestyle="--", linewidth=1.5,
            label=f"Sentinel ({SENTINEL_VALUE})")
ax1.set_xlabel("Value")
ax1.set_ylabel("Count")
ax1.set_title("BEFORE — with sentinel −0.01", fontweight="bold")
ax1.legend(fontsize=9)

ax2.hist(after_data, bins=50, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
ax2.set_xlabel("Value")
ax2.set_title("AFTER — sentinel replaced with 0.0", fontweight="bold")

fig.suptitle("Sentinel Replacement Effect — day_0_perceived_exertion",
             fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()
sns.despine()

save_figure(fig, "03_sentinel_before_after", subdir="preprocessing", close=False)
plt.show()
plt.close(fig)

### Interpretation

The sentinel spike at −0.01 is cleanly absorbed into the zero bin. The rest of the
distribution is unchanged — no information is lost or distorted. The zero bin now
correctly represents both "no training occurred" (from volume features) and "no
subjective rating recorded" (from perceived features), which are semantically
equivalent: **no training = zero load**.

---

## 3. Train/Test Split (GroupShuffleSplit by Athlete)

The most critical preprocessing step: splitting data while **preserving athlete
boundaries** (ADR-006).

**Why GroupShuffleSplit?** If the same athlete appears in both training and test
sets, the model could learn athlete-specific baselines (e.g., "athlete X always
has high exertion") rather than generalizable injury predictors. GroupShuffleSplit
ensures all observations from one athlete appear in **exactly one split**.

- **Split ratio**: 80% train / 20% test (by athletes, not by rows)
- **Grouping variable**: `athlete_id`
- **Random state**: fixed at 42 for reproducibility

In [ ]:
# Split both datasets
day_train, day_test = split_train_test(df_day)
week_train, week_test = split_train_test(df_week)

for name, train, test in [("Day", day_train, day_test), ("Week", week_train, week_test)]:
    n_train_athletes = train[ATHLETE_ID_COL].nunique()
    n_test_athletes = test[ATHLETE_ID_COL].nunique()
    print(f"{name} approach:")
    print(f"  Train: {len(train):,} rows, {n_train_athletes} athletes")
    print(f"  Test:  {len(test):,} rows, {n_test_athletes} athletes")
    print(f"  Row ratio: {len(train) / (len(train) + len(test)):.1%} / {len(test) / (len(train) + len(test)):.1%}")
    print()

In [ ]:
# === CRITICAL CHECK: No athlete leakage ===
day_train_athletes = set(day_train[ATHLETE_ID_COL].unique())
day_test_athletes = set(day_test[ATHLETE_ID_COL].unique())
assert day_train_athletes.isdisjoint(day_test_athletes), "LEAK: day athletes in both splits!"

week_train_athletes = set(week_train[ATHLETE_ID_COL].unique())
week_test_athletes = set(week_test[ATHLETE_ID_COL].unique())
assert week_train_athletes.isdisjoint(week_test_athletes), "LEAK: week athletes in both splits!"

print("No athlete leakage detected in either approach.")
print(f"  Day — Train athletes: {sorted(day_train_athletes)[:5]}... | Test athletes: {sorted(day_test_athletes)[:5]}...")
print(f"  Week — Train athletes: {sorted(week_train_athletes)[:5]}... | Test athletes: {sorted(week_test_athletes)[:5]}...")

In [ ]:
# Visualization: athlete assignment to train/test (day approach)
athlete_obs = (
    df_day.groupby(ATHLETE_ID_COL)
    .size()
    .reset_index(name="n_obs")
    .sort_values("n_obs", ascending=True)
)
athlete_obs["split"] = athlete_obs[ATHLETE_ID_COL].apply(
    lambda x: "Train" if x in day_train_athletes else "Test"
)

fig, ax = plt.subplots(figsize=(12, 14))

colors = [PALETTE["primary"] if s == "Train" else PALETTE["secondary"]
          for s in athlete_obs["split"]]

ax.barh(range(len(athlete_obs)), athlete_obs["n_obs"], color=colors,
        edgecolor="white", height=0.7, linewidth=0.3)

# Legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=PALETTE["primary"], label=f"Train ({len(day_train_athletes)} athletes)"),
    Patch(facecolor=PALETTE["secondary"], label=f"Test ({len(day_test_athletes)} athletes)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

ax.set_xlabel("Number of observations")
ax.set_ylabel("Athlete (sorted by observation count)")
ax.set_title("Athlete Assignment to Train/Test Split — Day Approach", fontweight="bold")
ax.set_yticks(range(0, len(athlete_obs), 5))
sns.despine()

save_figure(fig, "03_train_test_athlete_assignment", subdir="preprocessing", close=False)
plt.show()
plt.close(fig)

In [ ]:
# Class balance in train vs test
print("Class balance — Day approach:")
for name, split in [("Train", day_train), ("Test", day_test)]:
    rate = split[INJURY_COL].mean() * 100
    n_pos = int(split[INJURY_COL].sum())
    print(f"  {name}: {rate:.2f}% positive (n={n_pos:,} / {len(split):,})")

print("\nClass balance — Week approach (continuous, before binarization):")
for name, split in [("Train", week_train), ("Test", week_test)]:
    mean_score = split[INJURY_COL].mean()
    above_thresh = (split[INJURY_COL] >= WEEK_INJURY_THRESHOLD).mean() * 100
    print(f"  {name}: mean injury score = {mean_score:.4f}, ≥{WEEK_INJURY_THRESHOLD} → {above_thresh:.2f}%")

In [ ]:
# GroupKFold preview: how the 5 CV folds partition training athletes
k_fold = get_group_kfold()
X_day_train, y_day_train, groups_day_train = get_feature_target_groups(
    day_train, day_feature_cols
)

fold_info = []
for fold_idx, (train_idx, val_idx) in enumerate(
    k_fold.split(X_day_train, y_day_train, groups_day_train)
):
    fold_groups = groups_day_train.iloc[val_idx]
    fold_y = y_day_train.iloc[val_idx]
    fold_info.append({
        "Fold": fold_idx + 1,
        "Train rows": len(train_idx),
        "Val rows": len(val_idx),
        "Val athletes": fold_groups.nunique(),
        "Val injury %": f"{fold_y.mean() * 100:.2f}%",
    })

fold_df = pd.DataFrame(fold_info)
print("GroupKFold (5-fold) — Day training data:\n")
print(fold_df.to_string(index=False))

### Interpretation

- **No athlete leakage**: train and test athlete sets are completely disjoint.
- **Row ratio varies**: because GroupShuffleSplit splits by athletes (not rows),
  the actual row ratio depends on how much data each athlete contributed. Athletes
  with more observations have a larger impact on the split proportions.
- **Class balance preserved**: the injury rate in train and test should be roughly
  similar. Small differences are expected due to the athlete-level splitting —
  if a high-injury-rate athlete lands in the test set, it shifts the test rate.
- **GroupKFold preview**: the 5 folds partition training athletes into validation
  groups. Injury rate varies across folds because athletes differ in injury
  susceptibility — this is expected and reflects real-world heterogeneity.

The test set is **held out** until final model evaluation (notebook 04/05).
During development, only the GroupKFold cross-validation on the training set
is used to select hyperparameters and compare models.

---

## 4. Week Target Binarization

The week-approach target is a **continuous injury score** (0.0 to ~1.5+) that must
be binarized for classification. Following the paper's convention (ADR-002), we use
**threshold = 0.5**: values ≥ 0.5 become 1 (injured), values < 0.5 become 0 (not injured).

The threshold sensitivity analysis in notebook 02 documented how class balance
shifts at different thresholds (0.1, 0.25, 0.5, 0.75, 1.0).

In [ ]:
# Binarize week target in both splits
week_train[INJURY_COL] = binarize_target(week_train[INJURY_COL])
week_test[INJURY_COL] = binarize_target(week_test[INJURY_COL])

print(f"Week target binarized at threshold {WEEK_INJURY_THRESHOLD}:\n")
for name, split in [("Train", week_train), ("Test", week_test)]:
    rate = split[INJURY_COL].mean() * 100
    n_pos = int(split[INJURY_COL].sum())
    print(f"  {name}: {rate:.2f}% positive (n={n_pos:,} / {len(split):,})")

# Verify binary values only
assert set(week_train[INJURY_COL].unique()).issubset({0, 1})
assert set(week_test[INJURY_COL].unique()).issubset({0, 1})
print("\nTarget is binary (0/1 only) — verified.")

In [ ]:
# Side-by-side class balance comparison: day vs week
day_pos_pct = day_train[INJURY_COL].mean() * 100
week_pos_pct = week_train[INJURY_COL].mean() * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3.5))

for ax, pos_pct, title in [
    (ax1, day_pos_pct, "Day Approach"),
    (ax2, week_pos_pct, "Week Approach"),
]:
    neg_pct = 100 - pos_pct
    labels = [f"No Injury ({neg_pct:.2f}%)", f"Injury ({pos_pct:.2f}%)"]
    colors = [INJURY_PALETTE[0], INJURY_PALETTE[1]]
    bars = ax.barh(labels, [neg_pct, pos_pct], color=colors, height=0.5, edgecolor="white")
    ax.set_xlabel("% of training observations")
    ax.set_title(title, fontweight="bold")
    ax.set_xlim(0, 110)
    for bar, pct in zip(bars, [neg_pct, pos_pct]):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                f"{pct:.2f}%", va="center", fontweight="bold", fontsize=9)

fig.suptitle("Class Balance Comparison (Training Sets)", fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()
sns.despine(left=True)

save_figure(fig, "03_class_balance_comparison", subdir="preprocessing", close=False)
plt.show()
plt.close(fig)

### Interpretation

Both approaches show **extreme class imbalance**, but the week approach is somewhat
less severe due to the binarization threshold capturing a broader injury definition.

Both will use **class weighting** as the primary strategy (ADR-003):
- Logistic Regression / Random Forest: `class_weight='balanced'`
- XGBoost: `scale_pos_weight = n_negative / n_positive`

This tells the model to penalize misclassified injuries more heavily, proportional
to their rarity — a crucial design choice when the minority class is the one we
actually care about predicting.

---

## 5. Scaling Strategy

Feature scaling is **model-dependent**:

| Model                   | Needs scaling? | Why                                                                                     |
|-------------------------|----------------|-----------------------------------------------------------------------------------------|
| **Logistic Regression** | Yes            | Coefficients are magnitude-sensitive; L2 regularization assumes standardized features   |
| **Random Forest**       | No             | Splits are order-based (rank comparisons), not magnitude-based                          |
| **XGBoost**             | No             | Same as RF — tree splits are invariant to monotonic feature transformations             |

**Our approach**: fit `StandardScaler` on training features only (to prevent data
leakage), save the fitted scaler to disk, and apply it **conditionally** in the
modeling notebooks — only for Logistic Regression. The saved Parquet files contain
**unscaled** data.

In [ ]:
# Fit scalers on training features only
day_scaler = fit_scaler(day_train[day_feature_cols])
week_scaler = fit_scaler(week_train[week_feature_cols])

print("Day scaler — fitted on training features:")
print(f"  Features: {len(day_scaler.mean_)}")
print(f"  Mean range: [{day_scaler.mean_.min():.4f}, {day_scaler.mean_.max():.4f}]")
print(f"  Scale range: [{day_scaler.scale_.min():.4f}, {day_scaler.scale_.max():.4f}]")
print("\nWeek scaler — fitted on training features:")
print(f"  Features: {len(week_scaler.mean_)}")
print(f"  Mean range: [{week_scaler.mean_.min():.4f}, {week_scaler.mean_.max():.4f}]")
print(f"  Scale range: [{week_scaler.scale_.min():.4f}, {week_scaler.scale_.max():.4f}]")

In [ ]:
# Demonstrate scaling effect: before/after for 3 representative features
demo_features = ["day_0_total_km", "day_0_perceived_exertion", "day_0_km_z3_4"]
day_train_scaled = transform_scaled(day_train, day_scaler, day_feature_cols)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, feat in enumerate(demo_features):
    # Raw (top row)
    ax_raw = axes[0, i]
    ax_raw.hist(day_train[feat], bins=50, color=PALETTE["neutral"], alpha=0.7, edgecolor="white")
    ax_raw.set_title(f"{feat.replace('day_0_', '')}\n(raw)", fontweight="bold", fontsize=10)
    if i == 0:
        ax_raw.set_ylabel("Count")

    # Scaled (bottom row)
    ax_scaled = axes[1, i]
    ax_scaled.hist(day_train_scaled[feat], bins=50, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
    ax_scaled.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax_scaled.set_title(f"{feat.replace('day_0_', '')}\n(scaled)", fontweight="bold", fontsize=10)
    if i == 0:
        ax_scaled.set_ylabel("Count")

fig.suptitle("Scaling Effect — Raw vs StandardScaler (Day Training Data)",
             fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()

save_figure(fig, "03_scaling_before_after", subdir="preprocessing", close=False)
plt.show()
plt.close(fig)

In [ ]:
# Verify no data leakage: test set stats should NOT be exactly mean=0, std=1
day_test_scaled = transform_scaled(day_test, day_scaler, day_feature_cols)
test_stats = day_test_scaled[demo_features].describe().loc[["mean", "std"]]
test_stats.columns = [c.replace("day_0_", "") for c in test_stats.columns]

print("Scaled TEST set statistics (should be close to 0/1 but NOT exact):\n")
print(test_stats.round(4).to_string())
print("\n→ Non-exact values confirm the scaler was fit on TRAIN data only (no leakage).")

### Interpretation

- **Scaling centers features at 0** and normalizes their spread to unit variance.
  This puts all features on the same scale — critical for Logistic Regression where
  coefficients and regularization penalties are magnitude-sensitive.
- **Shape preserved**: scaling is a linear transformation — it does not change the
  distribution shape (skewness, zero-inflation). The heavy zero-inflation in features
  like `km_z3_4` remains visible even after scaling.
- **No leakage verified**: the test set statistics are close to mean=0 / std=1 but
  not exact, confirming the scaler was fit exclusively on training data.

---

## 6. Save Processed Data

We persist the preprocessed splits and fitted scalers to `data/processed/` for use
in the modeling notebooks (04, 05).

| File                 | Format  | Contents                                  |
|----------------------|---------|-------------------------------------------|
| `day_train.parquet`  | Parquet | Day train — sentinels replaced, unscaled  |
| `day_test.parquet`   | Parquet | Day test — sentinels replaced, unscaled   |
| `week_train.parquet` | Parquet | Week train — target binarized, unscaled   |
| `week_test.parquet`  | Parquet | Week test — target binarized, unscaled    |
| `day_scaler.pkl`     | joblib  | StandardScaler fit on day train features  |
| `week_scaler.pkl`    | joblib  | StandardScaler fit on week train features |

**Why Parquet?** Preserves dtypes exactly (no int→float conversion), smaller file
size than CSV, and faster read/write. **Why unscaled?** Tree-based models don't
need scaling — the scalers are saved separately for conditional use with LogReg.

In [ ]:
# Save all splits and scalers
save_splits(day_train, day_test, prefix="day")
save_splits(week_train, week_test, prefix="week")
save_scaler(day_scaler, name="day_scaler")
save_scaler(week_scaler, name="week_scaler")

print(f"All artifacts saved to: {PROCESSED_DATA_DIR}")

In [ ]:
# Round-trip verification: load back and check integrity
day_train_loaded, day_test_loaded = load_splits(prefix="day")
week_train_loaded, week_test_loaded = load_splits(prefix="week")
day_scaler_loaded = load_scaler(name="day_scaler")
week_scaler_loaded = load_scaler(name="week_scaler")

# Shape checks
assert day_train_loaded.shape == day_train.shape, "Day train shape mismatch"
assert day_test_loaded.shape == day_test.shape, "Day test shape mismatch"
assert week_train_loaded.shape == week_train.shape, "Week train shape mismatch"
assert week_test_loaded.shape == week_test.shape, "Week test shape mismatch"

# No athlete leakage in loaded data
assert set(day_train_loaded[ATHLETE_ID_COL].unique()).isdisjoint(
    set(day_test_loaded[ATHLETE_ID_COL].unique())
), "Leakage in loaded day data!"

# Scaler consistency
np.testing.assert_array_almost_equal(day_scaler.mean_, day_scaler_loaded.mean_)
np.testing.assert_array_almost_equal(week_scaler.mean_, week_scaler_loaded.mean_)

print("Round-trip verification passed:")
print(f"  Day train:  {day_train_loaded.shape}")
print(f"  Day test:   {day_test_loaded.shape}")
print(f"  Week train: {week_train_loaded.shape}")
print(f"  Week test:  {week_test_loaded.shape}")
print(f"  Day scaler:  {len(day_scaler_loaded.mean_)} features")
print(f"  Week scaler: {len(week_scaler_loaded.mean_)} features")

---

## 7. Summary: Preprocessing Pipeline

### Pipeline flow

```
Day approach:
  data/raw/day_*.csv
    → load_day_data()           # rename columns, validate schema
    → handle_sentinel_values()  # −0.01 → 0.0 (ADR-007)
    → split_train_test()        # GroupShuffleSplit by athlete (ADR-006)
    → fit_scaler(train)         # StandardScaler, train only
    → save to data/processed/   # 2 Parquet + 1 scaler pkl

Week approach:
  data/raw/week_*.csv
    → load_week_data()          # rename columns, validate schema
    → split_train_test()        # GroupShuffleSplit by athlete (ADR-006)
    → binarize_target()         # threshold 0.5 (ADR-002)
    → fit_scaler(train)         # StandardScaler, train only
    → save to data/processed/   # 2 Parquet + 1 scaler pkl
```

### Verified invariants

1. **No sentinel values remain** in day data after replacement
2. **No athlete leakage** between train and test splits (both approaches)
3. **Week target is binary** (0/1 only) after binarization
4. **Scaler fit on train only** — test set statistics confirm no leakage
5. **Round-trip persistence** — loaded data matches saved data exactly

### What the modeling notebooks receive

- **Unscaled DataFrames** (Parquet) with all columns (features + metadata)
- **Fitted scalers** (joblib) to apply conditionally for Logistic Regression
- Feature columns extracted via `get_feature_columns(df)`
- Groups extracted via `get_feature_target_groups(df, feature_cols)`

### Next steps

→ **Notebook 04**: Modeling — day approach (Dummy, LogReg, RF, XGBoost with GroupKFold CV)
→ **Notebook 05**: Modeling — week approach (same pipeline on week data)

In [ ]:
# Final check: list saved artifacts with file sizes
print(f"Contents of {PROCESSED_DATA_DIR}:\n")
for f in sorted(PROCESSED_DATA_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:30s} {size_kb:8.1f} KB")